**Before run this file, plz activate venv1**

# Import dataset

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../"))

# Import directly from the data file, bypassing any broken lobby paths!
from amoc_analysis.data import load_clean_data

series_35n = load_clean_data()
print(series_35n)

# Part A

In [ ]:
# 1. Compute statistics dynamically via your analysis engine
from amoc_analysis.analysis import calculate_statistics
mean_val, std_val = calculate_statistics(series_35n)

print(f"[Calculated] Mean: {mean_val:.4f} PW, Std Dev: {std_val:.4f} PW")

# 2. Render the publication-ready plot via your plot engine
from amoc_analysis.plotting import plot_mht_timeseries
plot_mht_timeseries(series_35n, mean_val, std_val)

# Part B

In [ ]:
# 1. Execute the Tukey Low-pass filter (Window = 12 quarters ≈ 3 years)
from amoc_analysis.analysis import apply_tukey_filter, compute_spectral_analysis
from amoc_analysis.plotting import plot_filtered_comparison
from amoc_analysis.plotting import plot_power_spectrum

series_filt = apply_tukey_filter(series_35n, window_size=12)

# 2. Render Figure 1: Time Series Intercomparison (This part is perfectly fine!)
plot_filtered_comparison(series_35n, series_filt)

# 3. Perform Spectral Analysis & Check Parseval's Theorem on the Raw Series
spec_results = compute_spectral_analysis(series_35n, series_filt, nperseg=36)

print("=" * 60)
print("             PARSEVAL THEOREM VARIANCE BUDGET")
print("=" * 60)
print(f"Computed Sampling Frequency (fs): {spec_results['fs']:.4f} yr^-1")
print(f"Time-Domain Total Variance:       {spec_results['variance']:.6f} PW^2")
print(f"Frequency-Domain Integrated Power: {spec_results['integrated_power']:.6f} PW^2")
print(f"Residual Discrepancy (Error):     {abs(spec_results['variance'] - spec_results['integrated_power']):.2e} PW^2")
print("=" * 60)

# 4. Render the dual-line log-scale spectrum plot directly
plot_power_spectrum(spec_results['frequencies'], spec_results['psd_raw'], spec_results['psd_filt'])

# Part C - filter design

In [ ]:
from amoc_analysis.analysis import compute_filter_responses
from amoc_analysis.plotting import plot_filter_comparison

# 1. Run frequency response simulation
response_results = compute_filter_responses(window_size=12)

# 2. Render response plot
plot_filter_comparison(
    response_results['frequencies'], 
    response_results['magnitude_tukey'], 
    response_results['magnitude_boxcar']
)

# Confidence intervals

In [ ]:
from amoc_analysis.analysis import compute_confidence_interval
from amoc_analysis.plotting import plot_power_spectrum_with_ci

# 1. Get total data length dynamically from your series
n_total = len(series_35n)

# 2. Compute 95% bounds and degrees of freedom for raw spectrum
lower_b, upper_b, dof_val = compute_confidence_interval(
    spec_results['psd_raw'], nperseg=36, n_total=n_total, alpha=0.05
)

# 3. Render the masterpiece plot!
plot_power_spectrum_with_ci(
    spec_results['frequencies'], 
    spec_results['psd_raw'], 
    spec_results['psd_filt'], 
    lower_b, upper_b, dof_val
)